# Test de personalidad
## 05 · Base de datos

Proyecto de Sistemas Inteligentes · UNAM · Matemáticas Aplicadas y Computación

## Objetivo

Guardar todos los datos del proyecto en una **base de datos**, que es el paso 2 de la estructura que pidió la maestra. Una base de datos es un archivo donde la información se guarda ordenada en **tablas** (como hojas de Excel, pero conectadas entre sí y con reglas que evitan errores) y se consulta con un lenguaje llamado **SQL**.

Usamos **SQLite**, que guarda toda la base en un solo archivo (`Base_Datos_Personalidad.db`) y no necesita instalar ningún servidor.

## Qué contiene

| Tabla | Qué guarda |
|---|---|
| `personas` | Una fila por persona con las respuestas de los **tres formularios** unidos y su perfil |
| `variables_modelo` | Las 9 variables que usamos para predecir y el perfil |
| `diccionario_variables` | A qué pregunta corresponde cada columna |
| `reglas_perfil` | Las reglas con las que se definió el perfil |
| `bitacora_limpieza` | Cada paso de la limpieza de datos |
| `resultados_modelos` | Métricas de cada modelo |
| `resumen_modelado` | Prueba de azar y agrupamiento |
| `matriz_confusion` | Aciertos y errores del mejor modelo |
| `pesos_logistica` | Peso de cada variable en la regresión logística |

Además hay **vistas** (consultas guardadas): una por cada formulario (`v_formulario_personalidad`, `v_formulario_musica`, `v_formulario_relaciones`) y tres resúmenes para el tablero.

**Privacidad:** la base **no** tiene correos, números de cuenta ni marcas de tiempo; solo códigos anónimos (`U001`...).

## Archivos que necesita

Los generaron los cuadernos anteriores y la limpieza: `Datos_Con_Perfil.xlsx`, `Datos_Modelado.xlsx`, `Datos_Unidos.xlsx`, `Bitacora_Limpieza.xlsx` y `Resultados_Modelos.xlsx`. Si no están en Colab, la celda de carga abre una ventana donde puedes elegirlos **todos juntos**.

### Preparar el entorno
*Qué hace:* importa las herramientas (`sqlite3` ya viene con Python).

In [ ]:
import os
import re
import shutil
import sqlite3

import pandas as pd

try:
    from google.colab import files  # solo existe dentro de Google Colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

print("pandas", pd.__version__, "| SQLite", sqlite3.sqlite_version, "| Colab:", EN_COLAB)

### Cargar los archivos
*Qué hace:* revisa que estén los cinco archivos (si faltan, te pide subirlos) y lee las hojas que vamos a guardar.

In [ ]:
ARCHIVOS = [
    "Datos_Con_Perfil.xlsx", "Datos_Modelado.xlsx", "Datos_Unidos.xlsx",
    "Bitacora_Limpieza.xlsx", "Resultados_Modelos.xlsx",
]

faltan = [a for a in ARCHIVOS if not os.path.exists(a)]
if faltan and EN_COLAB:
    print("Sube estos archivos (puedes elegirlos todos juntos):", faltan)
    files.upload()
    faltan = [a for a in ARCHIVOS if not os.path.exists(a)]
if faltan:
    raise FileNotFoundError(f"Faltan estos archivos: {faltan}. Súbelos con su nombre exacto.")

personas = pd.read_excel("Datos_Con_Perfil.xlsx", sheet_name="DatosConPerfil")
reglas = pd.read_excel("Datos_Con_Perfil.xlsx", sheet_name="ReglasPerfil")
variables_modelo = pd.read_excel("Datos_Modelado.xlsx", sheet_name="DatosModelado")
diccionario = pd.read_excel("Datos_Unidos.xlsx", sheet_name="Diccionario")
bitacora = pd.read_excel("Bitacora_Limpieza.xlsx", sheet_name="Bitacora").rename(columns={"n": "numero"})
modelos = pd.read_excel("Resultados_Modelos.xlsx", sheet_name="Modelos")
resumen_modelado = pd.read_excel("Resultados_Modelos.xlsx", sheet_name="Resumen")
matriz = pd.read_excel("Resultados_Modelos.xlsx", sheet_name="MatrizConfusion")
pesos = pd.read_excel("Resultados_Modelos.xlsx", sheet_name="PesosLogistica")

print("Personas:", len(personas), "| Variables del modelo:", variables_modelo.shape[1] - 3)

## Crear la base de datos
*Qué hace:* crea el archivo `Base_Datos_Personalidad.db` y sus tablas. Cada tabla se crea con su **estructura definida** (tipos de dato, llave primaria y reglas), no solo pegando datos:

- `id_persona` es **llave primaria**: no puede repetirse.
- `perfil` solo acepta valores válidos (`CHECK`).
- `variables_modelo` está **ligada** a `personas` por `id_persona` (llave foránea): no puede existir una persona en el modelo que no esté en la tabla principal.

In [ ]:
RUTA_DB = "Base_Datos_Personalidad.db"
if os.path.exists(RUTA_DB):
    os.remove(RUTA_DB)  # se crea desde cero para que el resultado sea siempre el mismo

con = sqlite3.connect(RUTA_DB)
con.execute("PRAGMA foreign_keys = ON")


def tipo_sql(serie):
    if pd.api.types.is_integer_dtype(serie):
        return "INTEGER"
    if pd.api.types.is_float_dtype(serie):
        return "REAL"
    return "TEXT"


def crear_tabla(nombre, tabla, clave=None, reglas_extra=()):
    columnas = []
    for c in tabla.columns:
        definicion = f'"{c}" {tipo_sql(tabla[c])}'
        if c == clave:
            definicion += " PRIMARY KEY"
        columnas.append(definicion)
    columnas += list(reglas_extra)
    con.execute(f'CREATE TABLE "{nombre}" ({", ".join(columnas)})')
    tabla.to_sql(nombre, con, if_exists="append", index=False)
    print(f"Tabla {nombre}: {len(tabla)} filas, {len(tabla.columns)} columnas")


PERFILES_SQL = "CHECK (perfil IN ('Introvertido', 'No introvertido', 'Por revisar'))"

crear_tabla("personas", personas, clave="id_persona", reglas_extra=[PERFILES_SQL])
crear_tabla("variables_modelo", variables_modelo, clave="id_persona",
            reglas_extra=[PERFILES_SQL, "FOREIGN KEY (id_persona) REFERENCES personas (id_persona)"])
crear_tabla("diccionario_variables", diccionario)
crear_tabla("reglas_perfil", reglas)
crear_tabla("bitacora_limpieza", bitacora)
crear_tabla("resultados_modelos", modelos)
crear_tabla("resumen_modelado", resumen_modelado)
crear_tabla("matriz_confusion", matriz)
crear_tabla("pesos_logistica", pesos)
con.commit()

### Crear las vistas
*Qué hace:* crea **vistas**, que son consultas guardadas con nombre. Las tres primeras recuperan las respuestas de cada formulario por separado (a partir de la tabla `personas`); las otras tres son resúmenes que usará el tablero.

In [ ]:
def columnas(prefijo):
    return ", ".join(["id_persona"] + [c for c in personas.columns if c.startswith(prefijo)])


vistas = {
    "v_formulario_personalidad": f"SELECT {columnas('pers_')} FROM personas",
    "v_formulario_musica": f"SELECT {columnas('mus_')} FROM personas WHERE tiene_musica = 'Sí'",
    "v_formulario_relaciones": f"SELECT {columnas('amor_')} FROM personas WHERE tiene_amor = 'Sí'",
    "v_resumen_perfiles": (
        "SELECT perfil, COUNT(*) AS personas, "
        "ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM personas), 1) AS porcentaje "
        "FROM personas GROUP BY perfil"
    ),
    "v_perfil_por_genero": (
        "SELECT pers_genero AS genero, perfil, COUNT(*) AS personas "
        "FROM personas GROUP BY pers_genero, perfil"
    ),
    "v_variables_por_perfil": (
        "SELECT perfil, COUNT(*) AS personas, "
        "ROUND(AVG(rol_lider) * 100, 0) AS pct_lider, "
        "ROUND(AVG(reaccion_incomodidad) * 100, 0) AS pct_incomodidad, "
        "ROUND(AVG(problema_con_otros) * 100, 0) AS pct_pide_ayuda, "
        "ROUND(AVG(act_conversar) * 100, 0) AS pct_conversa, "
        "ROUND(AVG(mus_tiempo_horas), 2) AS horas_musica_promedio "
        "FROM variables_modelo WHERE perfil IN ('Introvertido', 'No introvertido') GROUP BY perfil"
    ),
}
for nombre, consulta in vistas.items():
    con.execute(f'CREATE VIEW "{nombre}" AS {consulta}')
con.commit()
print("Vistas creadas:", list(vistas))

## Comprobar que la base quedó bien
*Qué hace:* verifica que (1) cada tabla tenga las mismas filas que el archivo de origen, (2) la base no esté dañada, (3) las llaves foráneas sean correctas y (4) **no haya datos personales** (ni columnas ni valores que parezcan correos o números de cuenta).

In [ ]:
origen = {
    "personas": personas, "variables_modelo": variables_modelo, "diccionario_variables": diccionario,
    "reglas_perfil": reglas, "bitacora_limpieza": bitacora, "resultados_modelos": modelos,
    "resumen_modelado": resumen_modelado, "matriz_confusion": matriz, "pesos_logistica": pesos,
}
for tabla, datos in origen.items():
    en_base = con.execute(f'SELECT COUNT(*) FROM "{tabla}"').fetchone()[0]
    assert en_base == len(datos), f"{tabla}: {en_base} filas en la base y {len(datos)} en el archivo"
print("1) Las filas de cada tabla coinciden con los archivos de origen: OK")

assert con.execute("PRAGMA integrity_check").fetchone()[0] == "ok"
print("2) Integridad de la base: OK")

assert con.execute("PRAGMA foreign_key_check").fetchall() == []
print("3) Llaves foráneas: OK")

patron_columna = re.compile(r"correo|cuenta|marca temporal", re.IGNORECASE)
patron_dato = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+|^\d{9}$")
for tabla in origen:
    info = con.execute(f'PRAGMA table_info("{tabla}")').fetchall()
    assert not any(patron_columna.search(fila[1]) for fila in info), f"columna sospechosa en {tabla}"
for columna in personas.columns:
    textos = [v.strip() for v in personas[columna].dropna() if isinstance(v, str)]
    assert not any(patron_dato.search(v) for v in textos), f"dato personal en {columna}"
print("4) Sin correos, números de cuenta ni marcas de tiempo: OK")

## Consultas de ejemplo (SQL)
*Qué hace:* ejecuta cinco consultas para mostrar cómo se usa la base. Se guardan también en `consultas_ejemplo.sql`.

In [ ]:
CONSULTAS = {
    "¿Cuántas personas hay de cada perfil?":
        "SELECT * FROM v_resumen_perfiles",
    "¿Cómo se comportan las variables según el perfil?":
        "SELECT * FROM v_variables_por_perfil",
    "¿Cómo se reparte el perfil por género?":
        "SELECT genero, perfil, personas FROM v_perfil_por_genero WHERE perfil <> 'Por revisar'",
    "¿Qué modelo tuvo mejores resultados?":
        "SELECT modelo, ROUND(aciertos_balanceados, 2) AS aciertos_balanceados, ROUND(f1, 2) AS f1, "
        "ROUND(auc, 2) AS auc FROM resultados_modelos ORDER BY aciertos_balanceados DESC",
    "¿Qué personas No introvertidas son líderes o conversan con otros? (une dos tablas)":
        "SELECT p.id_persona, p.perfil, m.rol_lider, m.act_conversar FROM personas p "
        "JOIN variables_modelo m ON m.id_persona = p.id_persona "
        "WHERE p.perfil = 'No introvertido' AND (m.rol_lider = 1 OR m.act_conversar = 1) ORDER BY p.id_persona",
}

with open("consultas_ejemplo.sql", "w", encoding="utf-8") as archivo:
    for pregunta, consulta in CONSULTAS.items():
        archivo.write(f"-- {pregunta}\n{consulta};\n\n")

for pregunta, consulta in CONSULTAS.items():
    print("\n" + pregunta)
    display(pd.read_sql_query(consulta, con))

## Exportar para la nube
*Qué hace:* guarda cada tabla como archivo **CSV** en un `.zip`. Con esos CSV se puede subir la base a un servicio en la nube (por ejemplo BigQuery o Google Sheets) y conectarla a un tablero como Looker Studio. Después cierra la base y, en Colab, descarga los archivos.

In [ ]:
os.makedirs("exportes_csv", exist_ok=True)
tablas = [f[0] for f in con.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")]
for tabla in tablas:
    pd.read_sql_query(f'SELECT * FROM "{tabla}"', con).to_csv(f"exportes_csv/{tabla}.csv", index=False, encoding="utf-8")

shutil.make_archive("exportes_csv", "zip", "exportes_csv")
con.close()

print("Tablas exportadas:", tablas)
print("Archivos creados: Base_Datos_Personalidad.db, consultas_ejemplo.sql, exportes_csv.zip")
if EN_COLAB:
    for archivo in ("Base_Datos_Personalidad.db", "consultas_ejemplo.sql", "exportes_csv.zip"):
        files.download(archivo)

## Conclusiones

- La base de datos `Base_Datos_Personalidad.db` guarda las respuestas de **los tres formularios** (unidos por persona y anónimos), el perfil, las variables del modelo, la bitácora de limpieza y los resultados de los modelos.
- Está **bien estructurada**: llaves primarias, llave foránea entre `personas` y `variables_modelo`, y una regla que solo permite perfiles válidos. Se comprobó su integridad y que no contiene datos personales.
- Las **vistas** y las consultas SQL muestran cómo recuperar la información, y los resúmenes alimentan el tablero.
- Es una base **local** (un archivo). Para llevarla a la nube se exportan las tablas a CSV.

**Siguiente paso:** el tablero analítico.